## 0. 初始化 NPU 环境

> ⚠️ **必须在 `import torch` 之前执行此单元格！**
> Jupyter 内核不继承 shell 环境变量，缺这步会导致 `torch_npu` 加载失败。

In [ ]:
import os, subprocess, warnings
warnings.filterwarnings('ignore')

# 加载 CANN 环境变量（Jupyter 内核不继承 shell 环境，必须手动 source）
# 尝试多个可能的 CANN 安装路径
_cann_paths = [
    os.path.join(os.environ.get("ASCEND_TOOLKIT_HOME", ""), "set_env.sh"),
    "/home/developer/Ascend/cann-9.2.0/set_env.sh",
    "/usr/local/Ascend/ascend-toolkit/set_env.sh",
]
_cann = next((p for p in _cann_paths if p and os.path.exists(p)), None)
if _cann is None:
    raise FileNotFoundError("找不到 CANN set_env.sh，请确认 CANN 安装路径")

print(f"正在加载 CANN 环境: {_cann}")
result = subprocess.run(["bash", "-c", f"source {_cann} && env"], capture_output=True, text=True, timeout=10)
for line in result.stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v

# 屏蔽 torchaudio（与当前 PyTorch 版本不兼容，本教程不需要）
import sys
sys.modules['torchaudio'] = None

print(f"✅ CANN 环境已加载 (ASCEND_TOOLKIT_HOME={os.environ.get('ASCEND_TOOLKIT_HOME','?')})")

# 基于昇腾 NPU 的数学解题大模型微调实战

本项目演示如何在华为昇腾 NPU 上，使用 **LoRA 微调**技术对 **Qwen2.5-0.5B-Instruct** 大语言模型进行微调，使其成为解答小学数学题的"数学解题专家"。

## 1. 整体流程概览

本项目将完整走通在昇腾 NPU 上使用 LoRA 微调大语言模型的全流程：

```
数据准备 → 模型加载（NPU） → Token 级数据预处理 → LoRA 适配器注入 → 训练（BF16） → 推理验证 → 精度评测
```

下面逐步实现每个环节。

---
## 2. 环境准备与依赖安装

在开始之前，我们需要确保环境中安装了必要的 Python 库。本项目依赖以下核心组件：

| 库名 | 用途 |
|------|------|
| `torch` + `torch_npu` | PyTorch 深度学习框架 + 昇腾 NPU 适配 |
| `transformers` | Hugging Face 模型加载与训练 |
| `peft` | 参数高效微调（LoRA 等） |
| `modelscope` | 模型下载（魔搭社区） |
| `swanlab` | 训练过程可视化监控 |

如果你使用的是魔搭（ModelScope）在线环境，这些库通常已预装。否则请执行以下安装命令：

In [ ]:
# 安装所需依赖（如在魔搭环境可跳过）
!pip install transformers modelscope==1.35.4 peft swanlab torch_npu datasets huggingface_hub

### 2.1 验证 NPU 环境

运行以下代码，确认昇腾 NPU 可用。如果输出 NPU 设备信息，说明环境配置正确。

In [ ]:
import torch
import torch_npu  # 导入昇腾 NPU 支持

# 检查 NPU 是否可用
print(f"NPU 是否可用: {torch.npu.is_available()}")
print(f"NPU 设备数量: {torch.npu.device_count()}")
print(f"当前 NPU 设备: {torch.npu.current_device()}")
print(f"NPU 设备名称: {torch.npu.get_device_name(0)}")

> 💡 **小贴士**：`torch_npu` 是昇腾 NPU 的 PyTorch 适配层。导入它之后，PyTorch 就能自动识别并使用 NPU 进行计算，使用方式与 `torch.cuda` 非常相似。例如：
> - `torch.npu.set_device(0)` 等价于 `torch.cuda.set_device(0)`
> - `model.to("npu")` 等价于 `model.to("cuda")`

---
## 3. 数据集探索与分析

在训练模型之前，我们先来了解一下我们的数据集。数据的好坏直接决定了模型最终的效果。

### 3.1 加载训练数据

In [ ]:
import os
import json

# 设置 HuggingFace 镜像端点（国内环境推荐，必须在 import huggingface_hub 之前设置）
os.environ["HF_ENDPOINT"] = "https://hub.gitcode.com"

from huggingface_hub import hf_hub_download

# 从 Hugging Face Hub 下载数据集
dataset_path = hf_hub_download(
    repo_id="ink_polymer/math-solver",
    filename="train.json",
    repo_type="dataset",
)

# 加载训练数据
with open(dataset_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"训练集总条数: {len(train_data)}")
print(f"数据类型: {type(train_data)}")
print(f"单条数据类型: {type(train_data[0])}")

### 3.2 查看数据样本

让我们看看训练数据的前 3 条样本，了解每条数据的结构：

In [ ]:
# 打印前 3 条样本
for i, sample in enumerate(train_data[:3]):
    print(f"{'='*60}")
    print(f"样本 {i}:")
    print(f"  指令 (instruction): {sample['instruction']}")
    print(f"  题目 (question):    {sample['question']}")
    print(f"  答案 (answer):      {sample['answer']}")
    print()

### 3.3 数据结构解读

每条训练数据包含 4 个字段：

| 字段 | 含义 | 示例 |
|------|------|------|
| `id` | 题目编号 | `"0"` |
| `instruction` | 系统指令，告诉模型"你是谁、该怎么做" | `"这是小学数学1-6年级的校内题目...直接输出数字答案"` |
| `question` | 具体的数学题目 | `"食堂运来105千克的萝卜..."` |
| `answer` | 标准答案 | `"315"` |

这种 **instruction + question + answer** 的三元组结构是大语言模型微调的标准范式：
- `instruction` 定义了模型的**角色和行为规范**
- `question` 是模型的**输入**
- `answer` 是模型需要学会的**正确输出**

In [ ]:
# 统计答案长度分布
answer_lengths = [len(str(item['answer'])) for item in train_data]
question_lengths = [len(item['question']) for item in train_data]

print(f"题目长度统计:")
print(f"  最短: {min(question_lengths)} 字符")
print(f"  最长: {max(question_lengths)} 字符")
print(f"  平均: {sum(question_lengths)/len(question_lengths):.1f} 字符")
print()
print(f"答案长度统计:")
print(f"  最短: {min(answer_lengths)} 字符")
print(f"  最长: {max(answer_lengths)} 字符")
print(f"  平均: {sum(answer_lengths)/len(answer_lengths):.1f} 字符")
print()

# 查看一些答案样例
print("部分答案样例:")
for i in range(0, len(train_data), len(train_data)//10):
    print(f"  题目: {train_data[i]['question'][:40]}... → 答案: {train_data[i]['answer']}")

### 3.4 准备测试样例

为了验证模型微调后的效果，我们预先准备了几道测试题目。这些题目与训练集风格一致，但不会参与训练，专门用于检验模型的解题能力。

In [ ]:
# 内联测试样例（无需外部文件）
test_samples = [

    {"id": 0, "question": "91.64与7.36的和乘43.6与3.6的差，积是多少？", "instruction": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
    {"id": 1, "question": "书架上有两层书，共164本，如果从下层取出9本放到上层去，两层数的本书就相同，原来下层比上层多多少本书？", "instruction": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
    {"id": 2, "question": "两个老师带着30名同学在公园里划船，每条船最多坐3人，至少需多少条船？", "instruction": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
    {"id": 3, "question": "每盒蛋糕7.90元，50元最多可以买多少盒蛋糕?", "instruction": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
    {"id": 4, "question": "淘气在书店买了一本《童话故事》和一本《数学世界》共花了14.7元，如果一本《童话故事》5.9元，一本《数学世界》多少元？", "instruction": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
]

print(f"测试样例数量: {len(test_samples)}")
print(f"\n前 3 条测试样例:")
for sample in test_samples[:3]:
    print(f"  [{sample['id']}] {sample['question'][:50]}...")

---
## 4. 模型下载与加载

### 4.1 选择基座模型

我们选择 **Qwen2.5-0.5B-Instruct** 作为基座模型。选择理由：
- **体积小**（0.5B 参数）：适合在单张 NPU 上快速训练和推理
- **Instruct 版本**：已经经过指令微调，具备良好的指令遵循能力
- **Qwen 系列**：在中文理解和数学推理方面表现优秀

### 4.2 下载模型

In [ ]:
from modelscope import snapshot_download

# 从魔搭社区下载 Qwen2.5-0.5B-Instruct 模型
# snapshot_download 会自动缓存，重复运行不会重新下载
model_dir = snapshot_download(

    "Qwen/Qwen2.5-0.5B-Instruct",

    cache_dir="./",
    revision="master"

)

print(f"模型下载完成，存放路径: {model_dir}")

### 4.3 加载模型与分词器

下载完成后，我们使用 Hugging Face 的 `transformers` 库加载模型。这里有两个关键组件：

- **Tokenizer（分词器）**：将文字转化为模型能理解的数字（token）
- **Model（模型）**：实际执行推理的神经网络

In [ ]:
import torch
from modelscope import AutoTokenizer
from transformers import AutoModelForCausalLM

# 指定使用 NPU 设备
device = "npu"

torch.npu.set_device(0)

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(

    "./Qwen/Qwen2.5-0.5B-Instruct/",

    use_fast=False,
    trust_remote_code=True

)

# 加载模型，使用 bfloat16 精度（NPU 原生支持）
model = AutoModelForCausalLM.from_pretrained(

    "./Qwen/Qwen2.5-0.5B-Instruct/",

    device_map=device,
    torch_dtype=torch.bfloat16

)

print(f"模型已加载到 {device} 设备")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

> 💡 **关键代码解读**：
> - `device_map="npu"`：告诉模型使用 NPU 而非 GPU
> - `torch_dtype=torch.bfloat16`：使用 BFloat16 精度，这是 NPU 的原生支持精度，相比 float32 可以节省一半显存，同时保持足够的数值精度
> - `trust_remote_code=True`：Qwen 模型包含自定义代码，需要信任远程代码

### 4.4 体验未微调的模型

在微调之前，让我们先看看原始模型解数学题的能力如何：

In [ ]:
# 用未微调的模型测试一道数学题
test_question = "食堂运来105千克的萝卜，运来的青菜是萝卜的3倍，运来青菜多少千克？"

messages = [

    {"role": "system", "content": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
    {"role": "user", "content": test_question}
]

# 将消息转化为模型可处理的格式
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(device)

# 生成回答
with torch.no_grad():
    generated_ids = model.generate(inputs.input_ids, max_new_tokens=128)

# 解码输出
generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(f"题目: {test_question}")
print(f"正确答案: 315")
print(f"模型回答: {response}")

可以看到，未经微调的模型可能会给出错误的答案，这正是我们需要通过微调来纠正的。

---
## 5. 数据预处理——将数学题转化为模型可学习的格式

### 5.1 为什么需要预处理？

大语言模型本质上是在学习"给定前面的文字，预测下一个 token"。因此我们需要将训练数据组织成特定的文本序列，让模型学会：

```
[系统指令] [用户问题] [模型应该输出的答案]
```

### 5.2 Qwen 的对话模板

Qwen2.5 使用特定的对话格式来区分不同角色的发言：

```
<|im_start|>system
这是小学数学1-6年级的校内题目...<|im_end|>
<|im_start|>user
食堂运来105千克的萝卜...<|im_end|>
<|im_start|>assistant
315
```

其中 `<|im_start|>` 和 `<|im_end|>` 是 Qwen 的特殊标记，用于标识每段对话的开始和结束。

### 5.3 数据预处理函数

下面的 `process_func` 函数负责将每条训练数据转化为模型可以学习的 token 序列：

In [ ]:
def process_func(example):

    """
    将单条训练数据转化为模型输入格式
    
    参数:
        example: 包含 instruction, question, answer 的字典
    
    返回:
        包含 input_ids, attention_mask, labels 的字典
    """

    MAX_LENGTH = 384  # 最大序列长度
    
    # Step 1: 将"系统指令 + 用户问题"编码为 input tokens
    # 这部分是模型的"输入"，模型不需要学习预测这部分
    instruction = tokenizer(

        f"<|im_start|>system\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>user\n{example['question']}<|im_end|>\n"
        f"<|im_start|>assistant\n",

        add_special_tokens=False,

    )

    # Step 2: 将"答案"编码为 response tokens
    # 这部分是模型需要学习预测的"输出"
    response = tokenizer(f"{example['answer']}", add_special_tokens=False)
    
    # Step 3: 拼接完整的 input_ids
    # 输入部分 + 回答部分 + 结束符
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    
    # Step 4: 构建 labels（标签）
    # 关键：输入部分的 label 设为 -100，表示"不计算这部分的损失"
    # 只有回答部分的 label 才参与损失计算
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]
    
    # Step 5: 截断超长序列
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {

        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

### 5.4 关键概念解读

上面的代码中有几个重要概念需要理解：

**① input_ids（输入 token 序列）**

分词器将文字转化为一串整数。例如 "你好" 可能被编码为 `[1083, 856]`。模型通过这些数字来"阅读"文字。

**② attention_mask（注意力掩码）**

标记哪些位置是有效的 token（值为 1），哪些是填充的（值为 0）。这帮助模型忽略无意义的填充位置。

**③ labels 中的 -100 技巧**

这是大语言模型微调中最重要的技巧之一：

```python
labels = [-100] * len(输入部分) + 回答部分的token_ids + [结束符]
```

- `-100` 是 PyTorch 中 `CrossEntropyLoss` 的**忽略索引**
- 输入部分（指令+问题）的 label 设为 -100 → 模型**不学习预测输入**
- 回答部分的 label 设为实际 token → 模型**只学习预测正确答案**

这就像考试：题目不需要模型去"预测"，模型只需要学会"写答案"。

### 5.5 执行数据预处理

In [ ]:
# 对整个训练集进行预处理
train_dataset = []
for d in train_data:
    train_dataset.append(process_func(d))

print(f"预处理完成，共 {len(train_dataset)} 条训练样本")
print(f"\n第一条样本的 input_ids 长度: {len(train_dataset[0]['input_ids'])}")
print(f"第一条样本的 labels 中非-100的数量: {sum(1 for l in train_dataset[0]['labels'] if l != -100)}")

# 可视化第一条样本的 label 分布
labels = train_dataset[0]['labels']
ignore_count = sum(1 for l in labels if l == -100)
learn_count = sum(1 for l in labels if l != -100)
print(f"\nLabel 分布: 忽略部分(不学习)={ignore_count} 个token, 学习部分(答案)={learn_count} 个token")

---
## 6. LoRA 微调配置与原理简介

### 6.1 什么是 LoRA？

**LoRA（Low-Rank Adaptation）** 是一种参数高效微调技术。它的核心思想是：

> 不修改原始模型的所有参数，而是在模型的关键层旁边"插入"一些小型的可训练矩阵（低秩矩阵），只训练这些新增的参数。

用一个比喻来理解：
- **全量微调**：把整栋房子推倒重建 → 成本高、耗时长
- **LoRA 微调**：在原有房子上加装几个智能模块 → 成本低、效果好

LoRA 的优势：
- 🚀 **训练参数少**：只需训练原模型 ~1% 的参数
- 💾 **显存占用低**：大幅降低显存需求
- ⚡ **训练速度快**：参数少自然训练更快
- 🔄 **可插拔**：不同的 LoRA 权重可以随时切换

### 6.2 LoRA 配置

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# 配置 LoRA 参数
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,          # 任务类型：因果语言模型（生成式）
    target_modules=[                        # 要在哪些层上应用 LoRA

        "q_proj", "k_proj", "v_proj", "o_proj",  # 注意力机制的 Q/K/V/O 投影层
        "gate_proj", "up_proj", "down_proj"        # 前馈网络（FFN）的投影层
    ],

    inference_mode=False,                   # 训练模式（非推理模式）
    r=8,                                    # LoRA 秩：低秩矩阵的维度
    lora_alpha=32,                          # LoRA 缩放因子
    lora_dropout=0.1,                       # Dropout 比例，防止过拟合

)

print("LoRA 配置:")
print(f"  任务类型: {config.task_type}")
print(f"  目标模块: {config.target_modules}")
print(f"  LoRA 秩 (r): {config.r}")
print(f"  LoRA Alpha: {config.lora_alpha}")
print(f"  Dropout: {config.lora_dropout}")

### 6.3 关键参数解读

| 参数 | 值 | 含义 |
|------|-----|------|
| `target_modules` | 7 个投影层 | 在 Transformer 的注意力层和前馈层上应用 LoRA |
| `r=8` | 秩为 8 | 低秩矩阵的维度，越大表达能力越强但参数越多 |
| `lora_alpha=32` | 缩放因子 32 | 控制 LoRA 更新的强度，通常设为 r 的 2~4 倍 |
| `lora_dropout=0.1` | 10% Dropout | 训练时随机丢弃 10% 的 LoRA 参数，防止过拟合 |

### 6.4 应用 LoRA 到模型

In [ ]:
# 重新加载模型（确保从原始权重开始）
model = AutoModelForCausalLM.from_pretrained(

    "./Qwen/Qwen2.5-0.5B-Instruct/",

    device_map=device,
    torch_dtype=torch.bfloat16

)

# 开启梯度检查点（节省显存）
model.enable_input_require_grads()
model = model.to(device)

# 应用 LoRA 配置
model = get_peft_model(model, config)

# 打印可训练参数信息
model.print_trainable_parameters()

可以看到，通过 LoRA 我们只需要训练模型总参数的 **不到 2%**，大大降低了训练成本。

---
## 7. 在昇腾 NPU 上启动训练

### 7.1 配置训练参数

Hugging Face 的 `TrainingArguments` 控制训练过程的所有超参数：

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./output/Qwen",             # 模型输出目录
    per_device_train_batch_size=64,         # 每张 NPU 上的 batch size
    gradient_accumulation_steps=1,          # 梯度累积步数（等效 batch_size = 64）
    logging_steps=10,                       # 每 10 步打印一次训练日志
    num_train_epochs=5,                     # 训练轮数（整个数据集遍历 5 次）
    save_steps=1000,                        # 每 1000 步保存一次 checkpoint
    learning_rate=1e-4,                     # 学习率
    save_on_each_node=True,                 # 每个节点都保存
    gradient_checkpointing=False,           # 0.5B 模型显存够用，关闭以加速
    report_to="none",                       # 不自动上报到第三方平台
    bf16=True,                              # 启用 BFloat16 训练（NPU 原生支持）
    dataloader_num_workers=4,               # 多线程数据加载（默认 0 是单线程，这是瓶颈！）
    dataloader_pin_memory=True,             # 锁页内存，加速 CPU→NPU 数据传输
)

print("训练参数配置:")
print(f"  输出目录: {args.output_dir}")
print(f"  Batch Size: {args.per_device_train_batch_size}")
print(f"  梯度累积步数: {args.gradient_accumulation_steps}")
print(f"  等效全局 Batch Size: {args.per_device_train_batch_size * args.gradient_accumulation_steps}")
print(f"  训练轮数: {args.num_train_epochs}")
print(f"  学习率: {args.learning_rate}")
print(f"  BFloat16: {args.bf16}")
print(f"  数据加载线程数: {args.dataloader_num_workers}")
print(f"  梯度检查点: {args.gradient_checkpointing}")

### 7.2 配置训练监控

使用 SwanLab 可以实时监控训练过程中的 loss 变化：

In [ ]:
import swanlab
from pathlib import Path

# 如果重复运行此 Cell，先结束上一个 run，避免 401 错误
try:
    swanlab.finish()
except Exception:
    pass

# ====== SwanLab 登录（已有凭证则跳过） ======
def _swanlab_credential_saved() -> bool:
    """检查本地是否已有 SwanLab 凭证"""
    for netrc_path in [Path.home() / ".netrc", Path.home() / ".swanlab" / ".netrc"]:
        if netrc_path.exists() and "swanlab" in netrc_path.read_text():
            return True
    return False

if _swanlab_credential_saved():
    print("检测到本地已有 SwanLab 凭证，跳过登录。")
else:
    import getpass
    api_key = getpass.getpass("请输入你的 SwanLab API Key（从 https://swanlab.cn/settings 获取）: ")
    swanlab.login(api_key=api_key, save=True)
    print("SwanLab 登录成功！凭证已保存，下次运行无需再输入。")


from swanlab.integration.transformers import SwanLabCallback

# 配置 SwanLab 训练监控
workspace = input("请输入 workspace（回车默认 npu-math-class）: ").strip() or "npu-math-class"
project = input("请输入 project（回车默认 math-solver-homework）: ").strip() or "math-solver-homework"
experiment_name = input("请输入姓名-学号（如: 张三-2024001）: ").strip()

swanlab_callback = SwanLabCallback(
    workspace=workspace,
    project=project,
    experiment_name=experiment_name,
    config={
        "model": "Qwen2.5-0.5B-Instruct",
        "lora_r": 8,
        "learning_rate": 1e-4,
        "batch_size": 64,
        "epochs": 5,
    }
)

print("SwanLab 监控已配置")

### 7.3 启动训练

一切准备就绪，现在创建 Trainer 并启动训练。这是整个课程中最核心的步骤：

In [ ]:
from transformers import Trainer, DataCollatorForSeq2Seq
import time

# 创建 Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)

print("开始训练...")
print(f"训练集大小: {len(train_dataset)} 条")
print(f"预计总步数: {len(train_dataset) // (args.per_device_train_batch_size * args.gradient_accumulation_steps) * args.num_train_epochs}")
print()

# 记时并启动训练
train_start = time.time()
trainer.train()
train_total_time = time.time() - train_start

# 记录性能指标到 SwanLab（用于排行）
total_samples = len(train_dataset) * args.num_train_epochs
throughput = total_samples / train_total_time  # 样本/秒

# NPU 显存使用
npu_mem = torch.npu.memory_allocated() / 1024 / 1024  # MB

swanlab.log({
    "perf/total_time_s": train_total_time,           # 总训练时间（秒）
    "perf/throughput_samples_per_s": throughput,      # 吞吐量（样本/秒）
    "perf/npu_memory_MB": npu_mem,                    # NPU 显存使用
    "perf/batch_size": args.per_device_train_batch_size,
})

print(f"\n🎉 训练完成！")
print(f"   总时间: {train_total_time:.1f}s")
print(f"   吞吐量: {throughput:.1f} 样本/秒")
print(f"   NPU 显存: {npu_mem:.0f} MB")

# 关闭 SwanLab
import swanlab
swanlab.finish()

### 7.4 训练过程解读

训练过程中你会看到类似如下的输出：

```
{'loss': 2.3456, 'learning_rate': 0.0001, 'epoch': 0.5}
{'loss': 1.2345, 'learning_rate': 0.0001, 'epoch': 1.0}
{'loss': 0.5678, 'learning_rate': 0.0001, 'epoch': 1.5}
...
```

- **loss（损失值）**：衡量模型预测与正确答案的差距。随着训练进行，loss 应该逐渐**下降**
- **learning_rate（学习率）**：控制每次参数更新的步长
- **epoch（轮次）**：当前训练进度，1.0 表示已经看完一遍全部数据

### 7.5 查看保存的 checkpoint

In [ ]:
import os

# 列出保存的模型检查点
output_dir = "./output/Qwen"
if os.path.exists(output_dir):
    checkpoints = sorted([d for d in os.listdir(output_dir) if d.startswith('checkpoint')])
    print(f"保存的检查点: {checkpoints}")
    
    # 查看最新检查点的内容
    if checkpoints:
        latest = os.path.join(output_dir, checkpoints[-1])
        print(f"\n最新检查点: {latest}")
        print(f"包含文件: {os.listdir(latest)}")

else:

    print("输出目录不存在，请先完成训练")

每个 checkpoint 中保存了：
- `adapter_model.safetensors`：LoRA 微调后的适配器权重（体积很小）
- `adapter_config.json`：LoRA 配置信息
- `optimizer.pt`、`scheduler.pt`：优化器和学习率调度器状态（用于断点续训）

## 8. 模型推理——让微调后的模型解题

训练完成后，我们用微调后的模型来解答数学题，验证学习效果。

In [ ]:
from transformers import AutoModelForCausalLM
from modelscope import AutoTokenizer
from peft import PeftModel
import torch
import torch_npu

device = "npu"

torch.npu.set_device(0)

# Step 1: 加载基座模型
print("正在加载基座模型...")
tokenizer = AutoTokenizer.from_pretrained(

    "./Qwen/Qwen2.5-0.5B-Instruct/",

    use_fast=False,
    trust_remote_code=True

)

model = AutoModelForCausalLM.from_pretrained(

    "./Qwen/Qwen2.5-0.5B-Instruct/",

    device_map=device,
    torch_dtype=torch.bfloat16

)

# Step 2: 叠加 LoRA 适配器（找到最新的 checkpoint）
# 自动找到最新的 checkpoint（按修改时间排序）
import os, glob
checkpoints = sorted(glob.glob("./output/Qwen/checkpoint-*"), key=os.path.getmtime)
if not checkpoints:
    raise FileNotFoundError("没有找到任何 checkpoint，请先完成训练")
checkpoint_path = checkpoints[-1]
print(f"加载最新 checkpoint: {checkpoint_path}")
print(f"正在加载 LoRA 适配器: {checkpoint_path}")
model = PeftModel.from_pretrained(model, model_id=checkpoint_path)
model = model.to(device)

print("✅ 微调模型加载完成！")

### 8.1 定义推理函数

封装一个通用的推理函数，方便后续批量调用：

In [ ]:
def predict(messages, model, tokenizer):

    """
    给定对话消息，使用模型生成回答
    
    参数:

        messages: 对话消息列表，格式为 [{"role": "system", "content": ...}, {"role": "user", "content": ...}]
        model: 加载好的模型
        tokenizer: 分词器

    返回:
        模型生成的回答文本
    """

    device = "npu"
    
    # 将消息转化为 Qwen 的对话模板格式
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True  # 在末尾添加 assistant 的起始标记

    )

    # 编码输入
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    # 生成回答
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512  # 最多生成 512 个 token

    )

    # 去掉输入部分，只保留新生成的 token
    generated_ids = [

        output_ids[len(input_ids):]

        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)

    ]

    # 解码为文本
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

### 8.2 单题测试

先用一道题验证模型是否正常工作：

In [ ]:
# 用内联测试样例验证模型
test_sample = test_samples[0]

messages = [
    {"role": "system", "content": test_sample['instruction']},
    {"role": "user", "content": test_sample['question']}
]

response = predict(messages, model, tokenizer)

print(f"题目: {test_sample['question']}")
print(f"模型回答: {response}")

# 再测试一道未微调时测试过的题目，对比微调前后的效果
compare_question = "食堂运来105千克的萝卜，运来的青菜是萝卜的3倍，运来青菜多少千克？"
compare_messages = [
    {"role": "system", "content": "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"},
    {"role": "user", "content": compare_question}
]

compare_response = predict(compare_messages, model, tokenizer)

print(f"题目: {compare_question}")
print(f"模型回答: {compare_response}")
print(f"正确答案: 315")


---

## 9. 交互式体验——与你的数学解题 AI 对话

最后，让我们用交互式的方式来体验微调后的模型。你可以输入任意的小学数学题，看看模型的解答能力如何。

In [ ]:
print("=" * 50)
print(" 数学解题 AI 助手（基于 Qwen2.5-0.5B + LoRA 微调）")
print(" 输入题目开始测试，输入 'q' 退出")
print("=" * 50)

instruction = "这是小学数学1-6年级的校内题目，无需进行分析，请直接输出数字答案，不带单位。"

# 预置几道测试题（也可以自己输入）
demo_questions = [

    "小明有12个苹果，吃了3个，又买了5个，现在有几个苹果？",
    "一个长方形的长是8厘米，宽是5厘米，周长是多少厘米？",
    "一列火车3小时行驶了240千米，火车的速度是多少千米/时？",
]

print("\n--- 自动演示 ---")
for q in demo_questions:
    messages = [

        {"role": "system", "content": instruction},
        {"role": "user", "content": q}
    ]

    response = predict(messages, model, tokenizer)
    print(f"\n题目: {q}")
    print(f"回答: {response}")

print("\n--- 演示结束 ---")